# Carbon mask on the LEGA-C DR3 window sample

- Mask: CN4150, G4300, Fe4383, C4668 bandpasses; `rest_range` 4074.5–4666.1 Å.
- References: `dr3_plain_w`, `dr3_cn_free_w` of `results/legac-dr3-2026-11-11`.

In [1]:
from pathlib import Path
import json
import sys
import h5py
import jax
import numpy as np
import pandas as pd
import astropy.units as u
from scipy.stats import linregress

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/experiment.py').exists())
sys.path.insert(0, str(ROOT))
from scripts.experiment import validate_result
from ceridwen.cosmology import age_gyr, resolve_cosmology
from ceridwen.observation.absorption_features import absorption_feature_mask, feature_windows

jax.config.update('jax_enable_x64', True)
OUT = ROOT / 'results/carbon-mask-window-2026-11-13'
DR3 = ROOT / 'results/legac-dr3-2026-11-11'
MASKED = ['CN4150', 'G4300', 'Fe4383', 'C4668']
W194 = json.loads((DR3 / 'targets.json').read_text())['targets']
ARMS = {'plain': DR3 / 'window-194/run/fits/dr3_plain_w', 'cn_free': DR3 / 'window-194/run/fits/dr3_cn_free_w',
        'mask': OUT / 'window-194/run/fits/dr3_mask_w'}
PARAMETERS = {'age_gyr': 'mass-weighted age [Gyr]', 'afe': 'afe', 'feh': 'logzsol'}
assert len(W194) == 194

- Summary quantiles of one fit; an invalid fit keeps its failure.
- `pull_rms`: rms pull over the fitted spectrum pixels.

In [2]:
def fit_row(directory):
    row = dict(path=str(directory.relative_to(ROOT)))
    try:
        validate_result(directory)
    except Exception as error:
        return row | dict(valid=False, failure=str(error).split(':')[0])
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        names = f['summary/parameter'].asstr()[:]
        q = {k: dict(zip(names, f[f'summary/{k}'][:])) for k in ('q16', 'q50', 'q84')}
        fitted = f['spectrum/mask'][:].astype(bool)
        row |= dict(valid=True, failure='', z=float(f.attrs['redshift']), seed=int(f.attrs['random_seed']),
                    fitted_pixels=int(fitted.sum()), pull_rms=float(np.sqrt(np.mean(f['spectrum/pull'][:][fitted] ** 2))),
                    floor_pct=float(q['q50']['calibration floor [%]']))
    for label, name in PARAMETERS.items():
        row |= {f'{label}_{k}': float(q[k][name]) for k in q}
    return row


def paired(new, old):
    """new − old posterior medians; σ is half the 16–84 range of old."""
    out = {}
    for label in PARAMETERS:
        delta = new[f'{label}_q50'] - old[f'{label}_q50']
        out |= {f'd_{label}': delta, f'd_{label}_sigma': delta / ((old[f'{label}_q84'] - old[f'{label}_q16']) / 2),
                f'{label}_width_ratio': (new[f'{label}_q84'] - new[f'{label}_q16']) / (old[f'{label}_q84'] - old[f'{label}_q16'])}
    return out

## Fitted pixels

- Per galaxy: fitted pixels of each arm, and the plain-arm pixels outside the masked bandpasses.
- Mg b bandpass (Trager et al. 1998, Table 2): fitted pixels in each arm.

In [3]:
def fitted_spectrum(directory):
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        return float(f.attrs['redshift']), f['spectrum/wavelength'][:].astype(float), f['spectrum/mask'][:].astype(bool)


rows = []
for t in W194:
    row = dict(target=t['spect_id'])
    for arm, root in ARMS.items():
        z, wave, fitted = fitted_spectrum(root / f"{t['object_id']}-{t['spect_id']}")
        low, high = feature_windows(['Mgb'], zred=z)[0]
        row |= {f'{arm}_fitted': int(fitted.sum()), f'{arm}_mgb_fitted': int((fitted & (wave >= low) & (wave <= high)).sum())}
        if arm == 'plain':
            row['plain_outside_mask'] = int((fitted & ~absorption_feature_mask(wave, zred=z, features=MASKED)).sum())
    rows.append(row)
pixels = pd.DataFrame(rows)
pixels.to_csv(OUT / 'fitted-pixels.csv', index=False)
pixels.drop(columns='target').describe().loc[['min', '50%', 'max']].T

,min,50%,max
plain_fitted,1286.0,1464.0,1817.0
plain_mgb_fitted,0.0,0.0,0.0
plain_outside_mask,893.0,1075.0,1322.0
cn_free_fitted,1286.0,1464.0,1817.0
cn_free_mgb_fitted,0.0,0.0,0.0
mask_fitted,893.0,1075.0,1322.0
mask_mgb_fitted,0.0,0.0,0.0


## Test set, 10 galaxies

- Mask arm against the plain DR3 window fit of the same galaxy (`test-set`, `dr3_window`).
- Δ = mask − plain median; σ = half the plain 16–84 range.

In [4]:
TEST = json.loads((OUT / 'test-10/experiment.json').read_text())['targets']
rows = []
for s in TEST:
    new = fit_row(next((OUT / 'test-10/run/fits/dr3_mask_w').glob(f'*-{s}')))
    old = fit_row(next((DR3 / 'test-set/run/fits/dr3_window').glob(f'*-{s}')))
    rows.append(dict(target=s, z=new['z'], valid=new['valid'], fitted_plain=old['fitted_pixels'], fitted_mask=new['fitted_pixels'],
                     pull_rms_plain=old['pull_rms'], pull_rms_mask=new['pull_rms'], floor_plain=old['floor_pct'], floor_mask=new['floor_pct'],
                     age_plain=old['age_gyr_q50'], age_mask=new['age_gyr_q50'], afe_plain=old['afe_q50'], afe_mask=new['afe_q50'],
                     feh_plain=old['feh_q50'], feh_mask=new['feh_q50']) | paired(new, old))
test = pd.DataFrame(rows)
test.to_csv(OUT / 'test-10.csv', index=False)
test.round(3)

,target,z,valid,fitted_plain,fitted_mask,pull_rms_plain,pull_rms_mask,floor_plain,floor_mask,age_plain,...,feh_mask,d_age_gyr,d_age_gyr_sigma,age_gyr_width_ratio,d_afe,d_afe_sigma,afe_width_ratio,d_feh,d_feh_sigma,feh_width_ratio
0,M1_210210,0.654,True,1392,973,1.053,1.048,2.295,2.369,5.302,...,-0.246,-0.435,-1.014,1.252,0.115,2.627,1.320,-0.004,-0.081,1.146
1,M8_150848,0.701,True,1434,1009,1.044,1.043,2.018,1.994,5.527,...,-0.277,-0.531,-1.082,1.109,0.130,2.206,1.014,0.051,0.902,1.101
2,M5_238314,0.733,True,1449,1124,1.101,1.126,3.659,3.411,4.441,...,-0.224,-0.339,-0.696,1.255,0.102,1.522,1.015,-0.011,-0.177,1.056
3,M3_120308,0.728,True,1455,1098,1.053,1.060,2.670,2.665,4.374,...,-0.300,-0.142,-0.294,1.121,0.087,1.257,1.146,-0.037,-0.646,1.199
4,M2_134021,0.749,True,1391,989,1.152,1.148,4.633,4.671,4.841,...,-0.466,-0.954,-1.847,1.386,0.354,3.081,0.983,-0.071,-0.749,1.286
5,M7_146213,0.687,True,1392,970,1.038,1.028,3.824,3.538,5.744,...,-0.253,-0.387,-0.751,1.193,0.001,0.018,1.122,0.132,1.523,1.141
6,M3_117010,0.677,True,1407,986,1.105,1.091,3.744,3.328,4.515,...,-0.235,-0.022,-0.037,1.324,-0.057,-0.806,1.226,-0.026,-0.342,1.227
7,M4_107370,0.844,True,1563,1134,1.115,1.140,3.779,3.564,3.171,...,-0.297,-0.201,-0.403,1.093,-0.052,-0.675,1.302,0.064,0.726,1.053
8,M14_37219,0.735,True,1470,1148,1.009,0.993,3.568,3.569,4.404,...,-0.264,-0.419,-0.769,1.110,0.008,0.091,1.134,0.032,0.374,0.993
9,M8_161113,0.891,True,1501,1129,1.058,1.066,3.699,3.478,4.301,...,-0.167,-0.255,-0.507,1.158,0.040,0.544,1.041,0.045,0.549,1.158


## Window sample, 194 galaxies

- Plain and C/N-free rows: `window-194-fits.csv` of the DR3 analysis.
- Mask rows: the same summary quantiles.

In [5]:
reference = pd.read_csv(DR3 / 'window-194-fits.csv')
reference = reference[reference.release == 'dr3']
GROUP = reference[reference.arm == 'plain'].set_index('target').group
mask_rows = pd.DataFrame([dict(arm='mask', release='dr3', target=t['spect_id'], object_id=t['object_id'], group=GROUP[t['spect_id']])
                          | fit_row(ARMS['mask'] / f"{t['object_id']}-{t['spect_id']}") for t in W194])
mask_rows.to_csv(OUT / 'window-194-mask-fits.csv', index=False)
fits = pd.concat([reference, mask_rows], ignore_index=True)
assert (fits[fits.valid].seed == 20260832).all()
fits.groupby('arm', sort=False).agg(fits=('target', 'size'), valid=('valid', 'sum'))

,fits,valid
arm,,
plain,194,194
cn_free,194,194
mask,194,194


- Slope: least-squares age against z; bootstrap 10000 galaxy resamples, seed 20261110.
- Planck18 line: same fit to the universe age at each galaxy's z.
- H = −1 / ((1 + z_med) dt/dz) (Jimenez & Loeb 2002, eq. 2).
- `line_fit`, `hubble`: as in the DR3 analysis notebook; its saved slopes are reproduced.

In [6]:
COSMO = resolve_cosmology(None)
H_UNIT = (1 / u.Gyr).to_value(u.km / u.s / u.Mpc)


def hubble(slope, z):
    return -H_UNIT / ((1 + z) * slope) if slope < 0 else np.nan


def line_fit(z, y):
    fit = linregress(z, y)
    indices = np.random.default_rng(20261110).integers(0, len(z), size=(10000, len(z)))
    zc = z[indices] - z[indices].mean(axis=1, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        draws = np.sum(zc * y[indices], axis=1) / np.sum(zc**2, axis=1)
    q16, q84 = np.nanquantile(draws, [0.16, 0.84])
    return dict(slope=fit.slope, q16=q16, q84=q84, scatter=np.std(y - fit.intercept - fit.slope * z, ddof=2), draws=draws)


GROUPS = {'all': lambda d: d, 'dr2_overlap': lambda d: d[d.group == 'dr2_overlap'], 'new_in_dr3': lambda d: d[d.group == 'new_in_dr3']}
rows = []
for arm, arm_fits in fits[fits.valid].groupby('arm', sort=False):
    for group, pick in GROUPS.items():
        g = pick(arm_fits)
        z, age = g.z.to_numpy(), g.age_gyr_q50.to_numpy()
        z_med = float(np.median(z))
        s, universe = line_fit(z, age), line_fit(z, np.asarray(age_gyr(z, COSMO)))
        rows.append(dict(arm=arm, group=group, galaxies=len(g), z_median=z_med, slope_gyr_per_z=s['slope'], bootstrap_q16=s['q16'],
                         bootstrap_q84=s['q84'], residual_scatter_gyr=s['scatter'], planck18_line_slope=universe['slope'],
                         slope_minus_planck_sigma=(s['slope'] - universe['slope']) / ((s['q84'] - s['q16']) / 2),
                         H_kms_mpc=hubble(s['slope'], z_med), H_from_q16=hubble(s['q16'], z_med), H_from_q84=hubble(s['q84'], z_med),
                         median_afe=g.afe_q50.median(), median_age_gyr=g.age_gyr_q50.median(), median_feh=g.feh_q50.median()))
slopes = pd.DataFrame(rows)
slopes.to_csv(OUT / 'age-redshift-slopes.csv', index=False)
saved = pd.read_csv(DR3 / 'window-194-age-redshift-slopes.csv').query("release == 'dr3'").set_index(['arm', 'group']).slope_gyr_per_z
ours = slopes.set_index(['arm', 'group']).slope_gyr_per_z
assert np.allclose(ours[saved.index], saved)
slopes.round(3)

,arm,group,galaxies,z_median,slope_gyr_per_z,bootstrap_q16,bootstrap_q84,residual_scatter_gyr,planck18_line_slope,slope_minus_planck_sigma,H_kms_mpc,H_from_q16,H_from_q84,median_afe,median_age_gyr,median_feh
0,plain,all,194,0.728,-5.043,-5.702,-4.388,0.875,-5.226,0.278,112.186,99.218,128.941,0.131,4.879,-0.317
1,plain,dr2_overlap,73,0.728,-4.766,-5.782,-3.723,0.942,-5.242,0.462,118.709,97.857,151.977,0.128,4.854,-0.319
2,plain,new_in_dr3,121,0.733,-5.203,-6.041,-4.392,0.838,-5.223,0.024,108.446,93.406,128.467,0.132,4.937,-0.312
3,cn_free,all,194,0.728,-5.154,-5.782,-4.534,0.805,-5.226,0.116,109.785,97.851,124.786,0.160,4.879,-0.325
4,cn_free,dr2_overlap,73,0.728,-5.018,-5.961,-4.067,0.818,-5.242,0.236,112.759,94.923,139.126,0.167,4.855,-0.330
5,cn_free,new_in_dr3,121,0.733,-5.271,-6.077,-4.489,0.800,-5.223,-0.060,107.058,92.851,125.710,0.158,4.898,-0.321
6,mask,all,194,0.728,-4.258,-4.898,-3.621,0.819,-5.226,1.516,132.889,115.510,156.237,0.229,4.575,-0.345
7,mask,dr2_overlap,73,0.728,-3.926,-4.967,-2.940,0.787,-5.242,1.298,144.104,113.921,192.483,0.229,4.493,-0.356
8,mask,new_in_dr3,121,0.733,-4.433,-5.256,-3.632,0.842,-5.223,0.973,127.279,107.362,155.384,0.229,4.609,-0.338


- Paired change of the mask arm against each reference arm, same galaxy.
- σ = half the reference 16–84 range; width ratio = mask / reference.

In [7]:
by_arm = {arm: g.set_index('target') for arm, g in fits[fits.valid].groupby('arm')}
rows = []
for other in ('plain', 'cn_free'):
    common = by_arm['mask'].index.intersection(by_arm[other].index)
    d = pd.DataFrame([paired(by_arm['mask'].loc[t], by_arm[other].loc[t]) for t in common], index=common)
    for label in PARAMETERS:
        rows.append(dict(reference=other, parameter=label, galaxies=len(d), median_delta=d[f'd_{label}'].median(),
                         delta_q16=d[f'd_{label}'].quantile(0.16), delta_q84=d[f'd_{label}'].quantile(0.84),
                         median_delta_sigma=d[f'd_{label}_sigma'].median(), max_abs_delta_sigma=d[f'd_{label}_sigma'].abs().max(),
                         median_width_ratio=d[f'{label}_width_ratio'].median()))
changes = pd.DataFrame(rows)
changes.to_csv(OUT / 'paired-changes.csv', index=False)
changes.round(3)

,reference,parameter,galaxies,median_delta,delta_q16,delta_q84,median_delta_sigma,max_abs_delta_sigma,median_width_ratio
0,plain,age_gyr,194,-0.297,-0.647,0.010,-0.540,9.450,1.122
1,plain,afe,194,0.080,-0.003,0.147,1.099,5.547,1.197
2,plain,feh,194,-0.033,-0.113,0.029,-0.427,43.472,1.161
3,cn_free,age_gyr,194,-0.331,-0.683,0.030,-0.623,5.748,1.106
4,cn_free,afe,194,0.068,-0.043,0.154,0.795,4.822,1.009
5,cn_free,feh,194,-0.027,-0.106,0.038,-0.362,5.197,1.162


- Slope difference, mask − reference, on identical galaxy resamples.
- σ = half the 16–84 range of the paired difference.

In [8]:
rows = []
for group, pick in GROUPS.items():
    targets = pick(by_arm['mask'].reset_index()).target
    z = by_arm['mask'].loc[targets, 'z'].to_numpy()
    s = {arm: line_fit(z, by_arm[arm].loc[targets, 'age_gyr_q50'].to_numpy()) for arm in by_arm}
    for other in ('plain', 'cn_free'):
        delta = s['mask']['slope'] - s[other]['slope']
        q16, q84 = np.nanquantile(s['mask']['draws'] - s[other]['draws'], [0.16, 0.84])
        rows.append(dict(group=group, reference=other, galaxies=len(z), slope_difference=delta, paired_q16=q16, paired_q84=q84,
                         difference_sigma=delta / ((q84 - q16) / 2)))
differences = pd.DataFrame(rows)
differences.to_csv(OUT / 'slope-differences.csv', index=False)
differences.round(3)

,group,reference,galaxies,slope_difference,paired_q16,paired_q84,difference_sigma
0,all,plain,194,0.786,0.507,1.069,2.795
1,all,cn_free,194,0.896,0.606,1.195,3.044
2,dr2_overlap,plain,73,0.840,0.387,1.245,1.959
3,dr2_overlap,cn_free,73,1.091,0.592,1.553,2.271
4,new_in_dr3,plain,121,0.770,0.424,1.132,2.175
5,new_in_dr3,cn_free,121,0.837,0.477,1.215,2.268
